# 🎙️ Garhwali TTS Training with Coqui

Fine-tune VITS model on Garhwali audio for Ghughuti AI voice synthesis.

**Prerequisites:**
- 50-100+ audio recordings from pahaditube.in/voice-recording
- Google Colab with GPU runtime (T4 free tier)

**Steps:**
1. Install dependencies
2. Download your recordings
3. Prepare dataset
4. Fine-tune VITS model
5. Test & export

## 1️⃣ Setup & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install Coqui TTS
!pip install -q TTS

# Additional dependencies
!pip install -q librosa soundfile pydub

print("✅ Installation complete!")

In [ ]:
# Verify GPU and imports
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ Download Recordings from PahadiTube

In [ ]:
import os
import requests
import json

# Create dataset directories
os.makedirs('/content/garhwali_dataset/wavs', exist_ok=True)

# PahadiTube API endpoint (replace with your actual domain)
API_URL = "https://pahaditube.in/api/tts"

# Fetch recordings metadata
print("Fetching recordings from PahadiTube...")
try:
    response = requests.get(f"{API_URL}/recordings")
    data = response.json()
    recordings = data.get('recordings', [])
    print(f"Found {len(recordings)} recordings")
except Exception as e:
    print(f"Error fetching recordings: {e}")
    print("\nManual upload instructions:")
    print("1. Download your recordings from the admin panel")
    print("2. Upload the ZIP file using the file upload below")
    recordings = []

In [ ]:
# Download audio files
from pydub import AudioSegment
import soundfile as sf
import numpy as np

metadata_lines = []
downloaded = 0

for rec in recordings:
    try:
        # Download audio
        audio_response = requests.get(rec['audio_url'] if 'audio_url' in rec else rec['url'])
        
        # Save temporarily as webm
        temp_path = f"/content/temp_{rec['sentenceId']}.webm"
        with open(temp_path, 'wb') as f:
            f.write(audio_response.content)
        
        # Convert to WAV (22050 Hz, mono)
        audio = AudioSegment.from_file(temp_path)
        audio = audio.set_frame_rate(22050).set_channels(1)
        
        wav_path = f"/content/garhwali_dataset/wavs/{rec['sentenceId']}.wav"
        audio.export(wav_path, format='wav')
        
        # Add to metadata
        metadata_lines.append(f"{rec['sentenceId']}|{rec['text']}")
        downloaded += 1
        
        # Cleanup
        os.remove(temp_path)
        
        if downloaded % 10 == 0:
            print(f"Downloaded {downloaded}/{len(recordings)}...")
            
    except Exception as e:
        print(f"Error processing {rec.get('sentenceId', 'unknown')}: {e}")

# Save metadata.csv
with open('/content/garhwali_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(metadata_lines))

print(f"\n✅ Downloaded {downloaded} recordings")
print(f"📄 Metadata saved to /content/garhwali_dataset/metadata.csv")

### Alternative: Upload ZIP manually

If API download doesn't work, upload a ZIP file with:
```
garhwali_dataset/
├── wavs/
│   ├── garhwali_001.wav
│   └── ...
└── metadata.csv
```

In [ ]:
# Upload ZIP manually (run this cell, then use the upload button)
from google.colab import files

print("Upload your dataset ZIP file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        !unzip -o "{filename}" -d /content/
        print(f"✅ Extracted {filename}")

## 3️⃣ Prepare Dataset & Config

In [ ]:
# Verify dataset
import os

wav_dir = '/content/garhwali_dataset/wavs'
metadata_file = '/content/garhwali_dataset/metadata.csv'

# Count files
wav_files = [f for f in os.listdir(wav_dir) if f.endswith('.wav')]
print(f"WAV files: {len(wav_files)}")

# Check metadata
with open(metadata_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()
print(f"Metadata entries: {len(lines)}")

# Show sample
print("\nSample entries:")
for line in lines[:5]:
    print(f"  {line.strip()}")

In [ ]:
# Calculate total audio duration
import librosa

total_duration = 0
for wav_file in wav_files:
    try:
        y, sr = librosa.load(os.path.join(wav_dir, wav_file), sr=None)
        total_duration += len(y) / sr
    except:
        pass

print(f"Total audio duration: {total_duration/60:.1f} minutes")
print(f"Average per clip: {total_duration/len(wav_files):.1f} seconds")

if total_duration < 30 * 60:
    print("\n⚠️ Warning: Less than 30 minutes of audio. Quality may be limited.")
    print("Consider recording more sentences at pahaditube.in/voice-recording")

## 4️⃣ Fine-tune VITS Model

In [ ]:
# Create training config
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.datasets import load_tts_samples

# VITS configuration for Garhwali
config = VitsConfig(
    run_name="garhwali-vits",
    run_description="Garhwali TTS fine-tuned from LJSpeech VITS",
    
    # Audio settings
    audio={
        "sample_rate": 22050,
        "win_length": 1024,
        "hop_length": 256,
        "num_mels": 80,
        "mel_fmin": 0,
        "mel_fmax": None,
    },
    
    # Training settings
    batch_size=16,  # Reduce if OOM
    eval_batch_size=8,
    num_loader_workers=4,
    num_eval_loader_workers=2,
    
    # Fine-tuning settings
    epochs=500,  # Start with 500, increase if quality is low
    save_step=500,
    save_checkpoints=True,
    print_step=50,
    plot_step=100,
    
    # Text processing - no phonemes for Garhwali
    use_phonemes=False,
    text_cleaner="basic_cleaners",
    
    # Dataset
    datasets=[{
        "formatter": "ljspeech",
        "meta_file_train": "metadata.csv",
        "path": "/content/garhwali_dataset/",
        "language": "hi",  # Use Hindi as base
    }],
    
    # Output
    output_path="/content/garhwali_output/",
)

# Save config
config.save_json("/content/garhwali_config.json")
print("✅ Config saved to /content/garhwali_config.json")

In [ ]:
# Download pre-trained VITS model for fine-tuning
from TTS.api import TTS

print("Downloading base VITS model (this may take a few minutes)...")
tts = TTS("tts_models/en/ljspeech/vits", progress_bar=True)
print("✅ Base model downloaded")

# Get model path
import os
model_path = os.path.expanduser("~/.local/share/tts/tts_models--en--ljspeech--vits")
print(f"Model path: {model_path}")

In [ ]:
# Start training
!tts --continue_path ~/.local/share/tts/tts_models--en--ljspeech--vits \
     --config_path /content/garhwali_config.json \
     --restore_path ~/.local/share/tts/tts_models--en--ljspeech--vits/model_file.pth

## 5️⃣ Test the Model

In [ ]:
# Find the best checkpoint
import os

output_dir = "/content/garhwali_output/"
checkpoints = [f for f in os.listdir(output_dir) if f.startswith('best_model')]

if checkpoints:
    best_model = os.path.join(output_dir, checkpoints[0])
    print(f"Best model: {best_model}")
else:
    # Use latest checkpoint
    checkpoints = sorted([f for f in os.listdir(output_dir) if f.endswith('.pth')])
    best_model = os.path.join(output_dir, checkpoints[-1]) if checkpoints else None
    print(f"Using checkpoint: {best_model}")

In [ ]:
# Test synthesis
from TTS.api import TTS
from IPython.display import Audio

# Load fine-tuned model
tts = TTS(
    model_path=best_model,
    config_path=os.path.join(output_dir, "config.json")
)

# Test sentences
test_sentences = [
    "नमस्कार, तुम कैसा छा?",
    "गढ़वाल बड़ो सुंदर छ",
    "आज मौसम बड़ो छ",
    "मी घर जान्दू",
]

for i, text in enumerate(test_sentences):
    output_path = f"/content/test_{i+1}.wav"
    tts.tts_to_file(text=text, file_path=output_path)
    print(f"\n{i+1}. {text}")
    display(Audio(output_path))

## 6️⃣ Export Model for Deployment

In [ ]:
# Package model for deployment
import shutil

export_dir = "/content/garhwali_tts_export"
os.makedirs(export_dir, exist_ok=True)

# Copy necessary files
shutil.copy(best_model, os.path.join(export_dir, "model.pth"))
shutil.copy(os.path.join(output_dir, "config.json"), export_dir)

# Create info file
info = {
    "name": "Garhwali VITS TTS",
    "language": "Garhwali (गढ़वाली)",
    "base_model": "tts_models/en/ljspeech/vits",
    "training_data": f"{len(wav_files)} recordings",
    "total_duration": f"{total_duration/60:.1f} minutes",
}

import json
with open(os.path.join(export_dir, "info.json"), "w") as f:
    json.dump(info, f, indent=2)

# Create ZIP
shutil.make_archive("/content/garhwali_tts_model", "zip", export_dir)
print("✅ Model exported to /content/garhwali_tts_model.zip")

In [ ]:
# Download the model
from google.colab import files
files.download("/content/garhwali_tts_model.zip")

## 📤 Next Steps: Deploy to HuggingFace

1. Go to [huggingface.co/new-space](https://huggingface.co/new-space)
2. Create a Gradio Space
3. Upload `garhwali_tts_model.zip`
4. Use the `app.py` from your `tts-training/huggingface/` folder

Or use the Hugging Face CLI:
```bash
pip install huggingface_hub
huggingface-cli login
huggingface-cli upload your-username/garhwali-tts ./garhwali_tts_export
```